## Cell 2 — Imports

In [ ]:
import os
import uuid
import json
import datetime

from dotenv import load_dotenv

from langchain_groq import ChatGroq
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage, trim_messages 

from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.sqlite import SqliteSaver
from langgraph.store.memory import InMemoryStore

from typing import TypedDict, List, Optional, Annotated
import operator

In [ ]:
load_dotenv()

GroqApiKey = os.getenv("GroqAPI")
ModelName = "llama-3.3-70b-versatile"
SqliteDbPath = "dual_memory.db"       
MaxActiveMessages = 10                  
DefaultUserId = "user_001"        

if not GroqApiKey:
    raise ValueError("GroqAPI not found in .env")

print(f"Model: {ModelName}")
print(f"SQLite DB: {SqliteDbPath}")
print(f"Max messages: {MaxActiveMessages}")
print(f"Default user: {DefaultUserId}")

In [ ]:
class AssistantState(TypedDict):

    Messages: Annotated[List, operator.add]
    Summary: str
    LongTermMemories : List[str]
    UserId: str

In [ ]:
LLM = ChatGroq(
    api_key = GroqApiKey,
    model = ModelName,
    temperature = 0.5,
)
LongTermStore = InMemoryStore()

In [ ]:
def loadLongTermMemoriesNode(State: AssistantState) -> dict:
   
    UserId = State.get("UserId", DefaultUserId)
    Namespace = (UserId, "facts") 

    StoredItems = LongTermStore.search(Namespace)
    MemoryList = []

    for Item in StoredItems:
        Fact = Item.value.get("fact", "")
        if Fact:
            MemoryList.append(Fact)

    if MemoryList:
        print(f"[LTM] Loaded {len(MemoryList)} long-term memories for {UserId}.")
    else:
        print(f"[LTM] No long-term memories found for {UserId}.")

    return {"LongTermMemories": MemoryList}

In [ ]:
def extractMemoriesNode(State: AssistantState) -> dict:
  
    UserId = State.get("UserId", DefaultUserId)
    Messages = State.get("Messages", [])

    LastHumanMessage = ""
    for Msg in reversed(Messages):
        if isinstance(Msg, HumanMessage):
            LastHumanMessage = Msg.content
            break

    if not LastHumanMessage:
        return {}

    ExtractionPrompt = [
        SystemMessage(content=(
            "You are a memory extractor. Read the user message and extract any personal facts worth remembering long-term. "
            "Examples: preferences, deadlines, names, roles, tools they use, goals. "
            "Return ONLY a valid JSON array of short fact strings. "
            "Return an empty array [] if no memorable facts are found. "
            "Do NOT include greetings, questions, or temporary context. "
            "Example output: [\"User prefers Python\", \"Deadline is Friday\", \"User's name is Alex\"]"
        )),
        HumanMessage(content=f"User message: {LastHumanMessage}")
    ]

    Response = LLM.invoke(ExtractionPrompt)
    RawOutput = Response.content.strip()

    ExtractedFacts = []
    try:
        CleanOutput = RawOutput.replace("```json", "").replace("```", "").strip()
        ExtractedFacts = json.loads(CleanOutput)

        if not isinstance(ExtractedFacts, list):
            ExtractedFacts = []

    except (json.JSONDecodeError, ValueError):
        ExtractedFacts = []

    Namespace = (UserId, "facts")

    for Fact in ExtractedFacts:
        if isinstance(Fact, str) and Fact.strip():
            
            MemoryKey = str(uuid.uuid4())[:8]
            LongTermStore.put(
                namespace = Namespace,
                key = MemoryKey,
                value = {
                    "fact": Fact.strip(),
                    "timestamp": datetime.datetime.now().isoformat(),
                    "source": LastHumanMessage[:100], 
                }
            )
            print(f"[LTM] Stored: {Fact.strip()}")

    return {}

In [ ]:
def summariseIfNeededNode(State: AssistantState) -> dict:
    
    Messages = State.get("Messages", [])
    ExistingSummary = State.get("Summary", "")

    if len(Messages) <= MaxActiveMessages:
        return {}

    print(f"[STM] {len(Messages)} messages, summarising overflow")

    SplitPoint = len(Messages) // 2
    OldMessages = Messages[:SplitPoint]
    KeepMessages = Messages[SplitPoint:]

    OldText = ""
    for Msg in OldMessages:
        if isinstance(Msg, HumanMessage):
            OldText += f"User: {Msg.content}\n"
        elif isinstance(Msg, AIMessage):
            OldText += f"Assistant: {Msg.content}\n"

    SummarisationMessages = [
        SystemMessage(content=(
            "You are a conversation summariser. "
            "Create a concise summary of the conversation below. "
            "Focus on key topics discussed, decisions made, and any important context. "
            "If there is an existing summary, merge the new content into it smoothly. "
            "Keep the final summary under 200 words."
        )),
        HumanMessage(content=(
            f"Existing summary:\n{ExistingSummary}\n\n"
            f"New messages to add:\n{OldText}"
        ))
    ]

    SummaryResponse = LLM.invoke(SummarisationMessages)
    NewSummary = SummaryResponse.content

    print(f"[STM] Summarised {len(OldMessages)} messages. Keeping {len(KeepMessages)} active.")

    return { "Summary": NewSummary, "TrimmedMessages" : KeepMessages, }

In [ ]:
def chatNode(State: AssistantState) -> dict:
    
    Messages = State.get("Messages", [])
    Summary = State.get("Summary", "")
    LongTermMemories = State.get("LongTermMemories", [])

    TrimmedMessages = State.get("TrimmedMessages", None)
    ActiveMessages = TrimmedMessages if TrimmedMessages else Messages

    SystemContent = "You are a helpful, knowledgeable personal assistant.\n"

    if LongTermMemories:
        SystemContent += "\nWhat I remember about you (long-term)\n"
        for Fact in LongTermMemories:
            SystemContent += f"- {Fact}\n"

    if Summary:
        SystemContent += "\nEarlier in this conversation (summary)\n"
        SystemContent += Summary + "\n"

    SystemContent += "\nRespond helpfully based on the above context and the conversation below."
    
    SafeMessages = trim_messages(
        ActiveMessages,
        max_tokens=MaxActiveMessages,
        strategy="last",
        token_counter=len,   
        include_system=False,
        allow_partial=False,
    )

    FullMessages = [SystemMessage(content=SystemContent)] + SafeMessages

    Response = LLM.invoke(FullMessages)
    BotReply = Response.content

    return {"Messages": [AIMessage(content=BotReply)]}

In [ ]:
class AssistantState(TypedDict):
    Messages: Annotated[List, operator.add]
    TrimmedMessages: Optional[List]                 
    Summary: str
    LongTermMemories: List[str]
    UserId: str

In [ ]:
def buildGraph(SqliteConn):
    
    Builder = StateGraph(AssistantState)
    ShortTermMemory = SqliteSaver(SqliteConn)

    Builder.add_node("load_ltm", loadLongTermMemoriesNode)
    Builder.add_node("extract_memories", extractMemoriesNode)
    Builder.add_node("summarise", summariseIfNeededNode)
    Builder.add_node("chat", chatNode)

    Builder.add_edge(START, "load_ltm")
    Builder.add_edge("load_ltm", "extract_memories")
    Builder.add_edge("extract_memories", "summarise")
    Builder.add_edge("summarise", "chat")
    Builder.add_edge("chat", END)

    Graph = Builder.compile(checkpointer=ShortTermMemory)

    return Graph

In [ ]:
def printAllMemories(UserId: str):
    
    Namespace = (UserId, "facts")
    StoredItems = LongTermStore.search(Namespace)

    print("\n\n")
    print(f"Long-Term Memories for: {UserId}")

    if not StoredItems:
        print("No memories stored yet.")
        print("Share personal facts and they will be remembered.")
    else:
        for Index, Item in enumerate(StoredItems, start=1):
            Fact = Item.value.get("fact", "(no fact)")
            Timestamp = Item.value.get("timestamp", "")[:19]
            print(f"{Index}. {Fact}")
            print(f"Stored: {Timestamp}")

def clearAllMemories(UserId: str):
    
    Namespace = (UserId, "facts")
    StoredItems = LongTermStore.search(Namespace)
    Count = 0

    for Item in StoredItems:
        LongTermStore.delete(Namespace, Item.key)
        Count += 1

    print(f"[LTM] Cleared {Count} memories for {UserId}.")

In [ ]:
def sendMessage(Graph, UserInput: str, ThreadId: str, UserId: str) -> str:
  
    Config = {"configurable": {"thread_id": ThreadId}}

    InputState = { "Messages": [HumanMessage(content=UserInput)], "UserId": UserId, }

    Result = Graph.invoke(InputState, config=Config)
    
    AllMessages = Result.get("Messages", [])
    if AllMessages:
        LastMessage = AllMessages[-1]
        if isinstance(LastMessage, AIMessage):
            return LastMessage.content

    return "(no response)"

In [ ]:
def runAssistant(UserId: str = DefaultUserId):
   
    import sqlite3

    print("\n\n")
    print("Dual-Memory Personal Assistant")
    print(f"User: {UserId}")
    print("Commands: /memories | /clearmemories | /newthread | quit")
    
    SqliteConn = sqlite3.connect(SqliteDbPath, check_same_thread=False)
    Graph = buildGraph(SqliteConn)

    ThreadId = f"{UserId}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
    print(f"Session thread: {ThreadId}")
    print("\n")

    while True:
        try:
            UserInput = input("You: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSession ended.")
            break

        if UserInput.lower() in ("quit", "exit", "q"):
            print("Goodbye!")
            break

        if not UserInput:
            continue

        if UserInput.lower() == "/memories":
            printAllMemories(UserId)
            continue

        if UserInput.lower() == "/clearmemories":
            clearAllMemories(UserId)
            continue

        if UserInput.lower() == "/newthread":
            ThreadId = f"{UserId}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
            print(f"[New thread started: {ThreadId}]")
            print("(Long-term memories are still active, only session history resets)")
            continue

        try:
            Reply = sendMessage(Graph, UserInput, ThreadId, UserId)
            print(f"\nAssistant: {Reply}\n")

        except Exception as Error:
            print(f"\n[Error]: {Error}")
            print("Please try again.\n")

    SqliteConn.close()

In [ ]:
import sqlite3

def runQuickTest():
    
    print("\n")
    print("Test: Dual Memory Assistant")
    print("\n")

    SqliteConn = sqlite3.connect(SqliteDbPath, check_same_thread=False)
    Graph = buildGraph(SqliteConn)

    TestUserId  = "test_user"
    TestThread  = "test_thread_001"

    TestMessages = [
        "My name is Haseeb and I prefer Python over JavaScript.",
        "My project deadline is Wednesday and I'm building a RAG chatbot.",
        "What were the key things you know about me so far?",
    ]

    for Msg in TestMessages:
        print(f"\nYou: {Msg}")
        Reply = sendMessage(Graph, Msg, TestThread, TestUserId)
        print(f"Assistant: {Reply}")

    print("\nLong-Term Memories Stored")
    printAllMemories(TestUserId)

    SqliteConn.close()

runQuickTest()

In [ ]:
runAssistant(UserId=DefaultUserId)